<a href="https://colab.research.google.com/github/Aditi-Nandiraju/Clever/blob/main/clever_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLEVER retrieval + repair on Colab (Lean + Mathlib + Ollama)

**First:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.

Logs, the LLM reply cache and results are kept in Google Drive (`MyDrive/clever-rr/`), so if
Colab disconnects, just run cells 1-5 again and re-run the experiment cell: finished problems are skipped.
Setup (cells 1-5) takes about 20-40 minutes on a fresh runtime.

## 1. Get the code, keep outputs in Drive

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/Aditi-Nandiraju/Clever.git /content/Clever || git -C /content/Clever pull -q
%cd /content/Clever

SAVE = '/content/drive/MyDrive/clever-rr'
for d in ['logs', 'cache', 'results']:
    os.makedirs(f'{SAVE}/{d}', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf {d}
        os.symlink(f'{SAVE}/{d}', d)
!ls -l

Mounted at /content/drive
/content/Clever
total 136
-rw-r--r-- 1 root root  9819 Nov 23 07:53 analyze.py
lrwxrwxrwx 1 root root    38 Nov 23 07:54 cache -> /content/drive/MyDrive/clever-rr/cache
-rw-r--r-- 1 root root  3819 Nov 23 07:53 checker.py
-rw-r--r-- 1 root root 35449 Nov 23 07:53 clever_colab.ipynb
-rw-r--r-- 1 root root  3515 Nov 23 07:53 config.py
-rw-r--r-- 1 root root  8430 Nov 23 07:53 corpus.py
drwxr-xr-x 2 root root  4096 Nov 23 07:53 data
-rw-r--r-- 1 root root  4649 Nov 23 07:53 data.py
-rw-r--r-- 1 root root  2017 Nov 23 07:53 errors.py
-rw-r--r-- 1 root root  4784 Nov 23 07:53 llm.py
lrwxrwxrwx 1 root root    37 Nov 23 07:54 logs -> /content/drive/MyDrive/clever-rr/logs
-rw-r--r-- 1 root root  4433 Nov 23 07:53 prompts.py
-rw-r--r-- 1 root root  7928 Nov 23 07:53 README.md
-rw-r--r-- 1 root root   253 Nov 23 07:53 requirements.txt
lrwxrwxrwx 1 root root    40 Nov 23 07:54 results -> /content/drive/MyDrive/clever-rr/results
-rw-r--r-- 1 root root  6273 Nov 23 07:53 r

## 2. Python packages

In [2]:
!pip install -q -r requirements.txt

## 3. Lean 4 + Mathlib
Installs elan, then the Lean version pinned by clever-bench (v4.24.0), downloads prebuilt
Mathlib and builds the CLEVER project. (`clever-bench-install` is not used: it does not
put Lean on the PATH inside Colab.)

In [3]:
import os
!curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
os.environ['PATH'] = os.path.expanduser('~/.elan/bin') + ':' + os.environ['PATH']

from clever_bench.benchmark import get_clever_lean_project_path
LEAN_DIR = str(get_clever_lean_project_path())
!cd {LEAN_DIR} && lake exe cache get && lake build 2>&1 | tail -n 5
!cd {LEAN_DIR} && lean --version

info: downloading installer
info: downloading https://releases.lean-lang.org/lean4/v4.24.0/lean-4.24.0-linux.tar.zst
438.3 MiB / 438.3 MiB (100 %)  38.3 MiB/s ETA:   0 s
info: installing /root/.elan/toolchains/leanprover--lean4---v4.24.0
info: mathlib: cloning https://github.com/leanprover-community/mathlib4
info: mathlib: checking out revision 'f897ebcf72cd16f89ab4577d0c826cd14afaafc7'
info: plausible: cloning https://github.com/leanprover-community/plausible
info: plausible: checking out revision 'dfd06ebfe8d0e8fa7faba9cb5e5a2e74e7bd2805'
info: LeanSearchClient: cloning https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision '99657ad92e23804e279f77ea6dbdeebaa1317b98'
info: importGraph: cloning https://github.com/leanprover-community/import-graph
info: importGraph: checking out revision 'd768126816be17600904726ca7976b185786e6b9'
info: proofwidgets: cloning https://github.com/leanprover-community/ProofWidgets4
info: proofwidgets: checking 

## 4. Ollama on the GPU
Same model as on the laptop: `qwen2.5:7b` with a 32k context (`qwen2.5-32k`, see `config.py`).

In [4]:
import subprocess, time
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
open('/content/Modelfile', 'w').write('FROM qwen2.5:7b\nPARAMETER num_ctx 32768\n')
!ollama pull qwen2.5:7b && ollama create qwen2.5-32k -f /content/Modelfile
!ollama list

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


NAME                  ID              SIZE      MODIFIED               
qwen2.5-32k:latest    891e0db9e750    4.7 GB    Less than a second ago    
qwen2.5:7b            845dbda0ea48    4.7 GB    Less than a second ago    


## 5. Retrieval corpus + offline check

In [5]:
!python corpus.py
!python smoke_test.py | tail -n 3
!python retriever.py 3

wrote /content/Clever/data/corpus.jsonl: 13180 lemmas, 8 examples
modules.json: 100% 349/349 [00:00<00:00, 1.52MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 457kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 31.7MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 344kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.88MB/s]

model.safetensors: downloading bytes:  91% 82.3M/90.9M [00:01<00:00, 94.0MB/s, 5.99MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:02<00:00, 41.9MB/s, 7.99MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:02<00:00, 44.8MB/s, 8.68MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 17651.93it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.96MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 76.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 112MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 710kB/s]
config.json: 100% 190/190 [00:00<00:00, 1.13MB/s]
Batches: 100% 104/104 [00:09<00:00, 

## 6. Real smoke test: 3 dev problems, real model + real Lean
Read a few records in the log before going further.

In [6]:
!python run_experiment.py --configs C1_baseline --split dev --limit 3 --seeds 0
!head -c 3000 logs/C1_baseline_s0_dev.jsonl


== C1_baseline seed 0 (dev): 0 problems to run
== C1_baseline seed 0 done in 0.0 min
{"config": "C1_baseline", "seed": 0, "pid": 0, "attempt": 1, "compiled": false, "proved": false, "stage": "impl", "feedback": "error at line 33 `def implementation (numbers: List Rat) (threshold: Rat) : Bool :=`: unexpected token 'import'; expected term\n\nerror at line 37 `import data.list.basic`: invalid 'import' command, it must be used in the beginning of the file", "raw_error": "ℹ [7351/7352] Replayed Imports.AllImports\ninfo: Imports/AllImports.lean:615:0: Unable to find a counter-example\ninfo: Imports/AllImports.lean:616:0: Unable to find a counter-example\ninfo: Imports/AllImports.lean:617:0: Unable to find a counter-example\ninfo: Imports/AllImports.lean:618:0: Unable to find a counter-example\ninfo: Imports/AllImports.lean:619:0: Unable to find a counter-example\ninfo: Imports/AllImports.lean:620:0: Unable to find a counter-example\ninfo: Imports/AllImports.lean:621:0: Unable to find a coun

## 7. Dev runs (tune prompts / error rules here, never on test)

In [7]:
!python run_experiment.py --split dev --seeds 0
!python analyze.py --split dev

Loading weights: 100% 103/103 [00:00<00:00, 5848.29it/s]
retriever: dense:sentence-transformers/all-MiniLM-L6-v2

== C1_baseline seed 0 (dev): 11 problems to run
  C1_baseline s0 p90  a1: parse
  C1_baseline s0 p90  a2: parse
  C1_baseline s0 p86  a1: impl
  C1_baseline s0 p86  a2: parse
  C1_baseline s0 p86  a3: parse
  C1_baseline s0 p90  a3: impl
  C1_baseline s0 p86  a4: parse
  C1_baseline s0 p90  a4: parse
  C1_baseline s0 p90  a5: parse
  C1_baseline s0 p86  a5: impl
  C1_baseline s0 p97  a1: parse
  C1_baseline s0 p105 a1: parse
  C1_baseline s0 p105 a2: impl
  C1_baseline s0 p97  a2: impl
  C1_baseline s0 p105 a3: impl
  C1_baseline s0 p97  a3: impl
  C1_baseline s0 p105 a4: impl
  C1_baseline s0 p97  a4: impl
  C1_baseline s0 p105 a5: impl
  [llm] ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) -> retry in 15s
  [llm] ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without 

In [9]:
import json, glob, collections
rows = [json.loads(l) for f in glob.glob('logs/*_dev.jsonl') for l in open(f)]
print(collections.Counter(r['stage'] for r in rows))
for stage in ['impl', 'parse']:
    for r in [r for r in rows if r['stage'] == stage][:3]:
        print(f"\n===== {stage}  p{r['pid']} a{r['attempt']}")
        print('FEEDBACK:', (r['feedback'] or '')[:500])
        if stage == 'parse':
            print('MODEL SAID:', open(glob.glob('cache/*.json')[0]).read()[:0] or '(see impl/proof fields)', r['impl'], r['proof'])


Counter({'impl': 43, 'parse': 15, 'proof': 6})

===== impl  p0 a1
FEEDBACK: error at line 33 `def implementation (numbers: List Rat) (threshold: Rat) : Bool :=`: unexpected token 'import'; expected term

error at line 37 `import data.list.basic`: invalid 'import' command, it must be used in the beginning of the file

===== impl  p11 a1
FEEDBACK: error at line 52 `#test implementation "010" "110" = "100"`: aborting evaluation since the expression depends on the 'sorry' axiom, which can lead to runtime instability and crashes.

To attempt to evaluate anyway despite the risks, use the '#eval!' command.

===== impl  p0 a2
FEEDBACK: error at line 39 `| _, _ when found => found`: Invalid pattern: Expected an identifier in function position, but found
  _

error at line 57 `#test implementation ([1, 2, 3]: List Rat) 0.5 = false`: aborting evaluation since the expression depends on the 'sorry' axiom, which can lead to runtime instability and crashes.

To attempt to evaluate anyway despite the 

## 8. Test runs (reported numbers). Freeze `config.py` and `prompts.py` first.
Long: re-run this cell after a disconnect, it resumes. Results land in Drive: `MyDrive/clever-rr/results/test/`.

In [8]:
!python run_experiment.py --configs C1_baseline C2_repair
!python run_experiment.py --configs C3_retrieval C4_both
!python analyze.py


== C1_baseline seed 0 (test): 40 problems to run
  [llm] ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConn -> retry in 15s
  [llm] ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConn -> retry in 15s
  [llm] ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConn -> retry in 15s
  [llm] ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConn -> retry in 15s
^C
^C
Traceback (most recent call last):
  File "/content/Clever/analyze.py", line 20, in <module>
    import matplotlib
  File "/usr/local/lib/python3.13/dist-packages/matplotlib/__init__.py", line 161, in <module>
    from . import _api, _version, cbook, _docstring, rcsetup
  File "/usr/local/lib/py